# SIEWS+ — Open Hole & Construction Safety
## YOLO26n Transfer Learning — 1 Dataset Murni

**5 Classes:** barricade (0), hard-hat (1), open-hole (2), safety-cone (3), vest (4)

**Kaggle:** Upload zip → Enable GPU T4 → Run All → Download `best_stage3_openhole.pt`

## 1. Setup

In [ ]:
import os, sys, subprocess, zipfile, shutil, random, yaml, traceback
from pathlib import Path
from collections import Counter

IS_KAGGLE = os.path.exists('/kaggle/input')
print(f'Kaggle: {IS_KAGGLE}')

subprocess.run([sys.executable,'-m','pip','install','-U','ultralytics','-q'], check=True)
import ultralytics, torch
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'

## 2. Path Config

In [ ]:
ZIP_NAME = 'Open hole Detection - Construction Safety.v14-final-training.yolo26.zip'

if IS_KAGGLE:
    # Kaggle otomatis ekstrak dataset — tidak ada .zip
    # Cari folder yang mengandung train/images
    DATASET_ZIP = None
    OUTPUT_DIR  = Path('/kaggle/working')
    EXTRACT_DIR = None  # akan ditentukan di cell berikutnya
else:
    DATASET_ZIP = Path('F:/migas-siews/dataset') / ZIP_NAME
    OUTPUT_DIR  = Path('F:/migas-siews/training/runs_openhole')
    EXTRACT_DIR = OUTPUT_DIR / 'dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if DATASET_ZIP:
    print(f'ZIP    : {DATASET_ZIP}')
    print(f'Exists : {DATASET_ZIP.exists()}')
else:
    print('Kaggle mode — dataset akan dicari di /kaggle/input/')

## 3. Ekstrak Dataset

In [ ]:
if IS_KAGGLE:
    # Di Kaggle: dataset sudah diekstrak oleh platform
    # Cari folder yang punya struktur train/images
    kaggle_input = Path('/kaggle/input')
    found = None
    for candidate in kaggle_input.rglob('train'):
        if (candidate / 'images').exists():
            found = candidate.parent
            break
    if found is None:
        # Fallback: ambil subfolder pertama dari /kaggle/input/
        subdirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
        print('Isi /kaggle/input/:', [d.name for d in subdirs])
        if subdirs:
            found = subdirs[0]
        else:
            raise FileNotFoundError(
                'Dataset tidak ditemukan!\n'
                'Pastikan sudah add dataset di Kaggle:\n'
                '  Notebook Settings → Add Data → upload ZIP'
            )
    EXTRACT_DIR = found
    print(f'Dataset ditemukan: {EXTRACT_DIR}')

elif DATASET_ZIP and DATASET_ZIP.exists():
    if not EXTRACT_DIR.exists():
        print('Extracting...')
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        print('Done!')
    else:
        print('Already extracted')

else:
    raise FileNotFoundError(f'Dataset ZIP tidak ada: {DATASET_ZIP}')

# Verifikasi struktur
print()
for split in ['train', 'valid', 'test']:
    p = EXTRACT_DIR / split / 'images'
    if p.exists():
        ni = len(list(p.glob('*.*')))
        nl = len(list((EXTRACT_DIR/split/'labels').glob('*.txt'))) if (EXTRACT_DIR/split/'labels').exists() else 0
        print(f'[{split:6s}] images={ni:5d}  labels={nl:5d}')
    else:
        print(f'[{split:6s}] NOT FOUND — cek struktur dataset')

## 4. Verifikasi Classes

In [ ]:
CLASS_NAMES = {
    0: 'barricade',
    1: 'hard-hat',
    2: 'safety-cone',   # class 2 = cone (bukan open-hole)
    3: 'open-hole',     # class 3 = lubang (bukan safety-cone)
    4: 'vest',
}
NC = len(CLASS_NAMES)

counts = Counter()
for lf in (EXTRACT_DIR/'train'/'labels').glob('*.txt'):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except: pass

print('Class distribution:')
for cid in sorted(counts):
    print(f'  [{cid}] {CLASS_NAMES.get(cid,"UNKNOWN"):15s}: {counts[cid]} instances')
max_id = max(counts) if counts else -1
print(f'Max class ID: {max_id} | NC: {NC}')
if max_id >= NC:
    print('Tambah class ke CLASS_NAMES!')
else:
    print('Class OK')

## 5. Buat data.yaml

In [ ]:
val_split  = 'valid' if (EXTRACT_DIR/'valid'/'images').exists() else 'val'
test_split = 'test'  if (EXTRACT_DIR/'test'/'images').exists()  else val_split

DATA_YAML = OUTPUT_DIR / 'openhole.yaml'
yaml.dump({
    'path' : str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : f'{test_split}/images',
    'nc'   : NC,
    'names': CLASS_NAMES,
}, open(DATA_YAML,'w'), default_flow_style=False)
print(DATA_YAML.read_text())

## 6. Sample Visualisasi

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
COLORS = [(255,100,0),(0,200,255),(255,0,0),(255,200,0),(0,255,100)]

train_imgs = list((EXTRACT_DIR/'train'/'images').glob('*.jpg'))
samples = random.sample(train_imgs, min(6,len(train_imgs)))
fig, axes = plt.subplots(2,3,figsize=(15,8))
for ax, ip in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    lp = EXTRACT_DIR/'train'/'labels'/(ip.stem+'.txt')
    names_found = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            p = line.split()
            if len(p)<5: continue
            c=int(p[0]); xc,yc,bw,bh=map(float,p[1:5])
            x1,y1=int((xc-bw/2)*w),int((yc-bh/2)*h)
            x2,y2=int((xc+bw/2)*w),int((yc+bh/2)*h)
            col=COLORS[c%len(COLORS)]
            cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
            nm=CLASS_NAMES.get(c,f'cls_{c}')
            cv2.putText(img,nm,(x1,max(y1-5,10)),cv2.FONT_HERSHEY_SIMPLEX,0.55,col,2)
            names_found.append(nm)
    ax.imshow(img); ax.axis('off'); ax.set_title(', '.join(set(names_found)) or '-', fontsize=9)
plt.suptitle('Sample Dataset — Open Hole Construction Safety', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'samples.png'), dpi=100)
plt.show()

## 7. Training

In [ ]:
from ultralytics import YOLO

EPOCHS   = 150   # naikkan dari 100 agar sempat explore lebih jauh
BATCH    = 16
IMGSZ    = 480
PATIENCE = 0     # 0 = disable EarlyStopping, train sampai selesai
FREEZE   = 10

RUN_DIR = OUTPUT_DIR/'runs'/'openhole'

print(f'epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'EarlyStopping: DISABLED (patience=0)')

TRAIN_OK = False
try:
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, patience=PATIENCE, freeze=FREEZE, pretrained=True,
        project=str(OUTPUT_DIR/'runs'), name='openhole', exist_ok=True,
        # Augmentasi konstruksi outdoor
        augment=True, mosaic=1.0, mixup=0.05, copy_paste=0.05,
        hsv_h=0.02, hsv_s=0.75, hsv_v=0.4,
        fliplr=0.5, flipud=0.0, degrees=8.0, scale=0.5, erasing=0.3,
        # Optimizer
        optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005, warmup_epochs=3,
        save_period=10, verbose=True, plots=True,
    )
    print('✅ Training selesai!')
    TRAIN_OK = True
except KeyboardInterrupt:
    print('⚠️  Dihentikan manual — checkpoint ada di:', RUN_DIR/'weights'/'last.pt')
except Exception as e:
    print(f'❌ Error: {e}'); traceback.print_exc()

## 8. Evaluasi

In [ ]:
best_pt = RUN_DIR/'weights'/'best.pt'
last_pt = RUN_DIR/'weights'/'last.pt'
eval_pt = best_pt if best_pt.exists() else last_pt

if not eval_pt.exists():
    print('❌ Tidak ada checkpoint!')
else:
    try:
        if TRAIN_OK:
            m = results.results_dict
            map50=m.get('metrics/mAP50(B)',0); prec=m.get('metrics/precision(B)',0); rec=m.get('metrics/recall(B)',0)
        else:
            em=YOLO(str(eval_pt)); vr=em.val(data=str(DATA_YAML),device=DEVICE,verbose=False)
            map50=vr.box.map50; prec=vr.box.mp; rec=vr.box.mr
        print(f'mAP50     : {map50:.4f}  {"✅" if map50>=0.80 else "⚠️" if map50>=0.60 else "❌"}')
        print(f'Precision : {prec:.4f}')
        print(f'Recall    : {rec:.4f}')
    except Exception as e:
        print(f'Eval error: {e}')

In [ ]:
from IPython.display import Image as IPImg, display
for pn in ['results.png','confusion_matrix.png','PR_curve.png']:
    pf = RUN_DIR/pn
    if pf.exists(): print(f'--- {pn} ---'); display(IPImg(str(pf),width=750))

## 9. Inference Test

In [ ]:
if eval_pt.exists():
    bm = YOLO(str(eval_pt))
    tdir = EXTRACT_DIR/'test'/'images'
    if not tdir.exists(): tdir = EXTRACT_DIR/val_split/'images'
    imgs = random.sample(list(tdir.glob('*.jpg')), min(6,len(list(tdir.glob('*.jpg')))))
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,ip in zip(axes.flat,imgs):
        try:
            pred=bm.predict(str(ip),conf=0.25,verbose=False)[0]
            ax.imshow(cv2.cvtColor(pred.plot(),cv2.COLOR_BGR2RGB))
            bxs=pred.boxes
            title=', '.join([f"{pred.names[int(b.cls[0])]}:{float(b.conf[0]):.0%}" for b in bxs]) if bxs and len(bxs) else 'no det'
            ax.set_title(title[:40],fontsize=8)
        except Exception as e: ax.set_title(str(e)[:30])
        ax.axis('off')
    plt.suptitle('Inference Test',fontsize=13); plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR/'inference.png'),dpi=100); plt.show()

## 10. Auto-Save Model

In [ ]:
dest = Path('/kaggle/working') if IS_KAGGLE else Path('F:/migas-siews/backend/models')
dest.mkdir(parents=True, exist_ok=True)

for src, name in [(best_pt,'best_stage3_openhole.pt'),(last_pt,'last_stage3_openhole.pt')]:
    if src.exists():
        dst = dest/name
        shutil.copy2(src, dst)
        print(f'✅ Saved: {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'⚠️  Tidak ada: {src}')

if IS_KAGGLE:
    print('\nLangkah selanjutnya:')
    print('  1. Download best_stage3_openhole.pt dari tab Output Kaggle')
    print('  2. Salin ke: f:/migas-siews/backend/models/')
    print('  3. Update detector.py S3: ganti ke best_stage3_openhole.pt')
    print('  4. Restart backend')

## 11. Saran jika mAP Rendah

| Masalah | Solusi |
|---|---|
| mAP < 0.60 | Kurangi `freeze=5` atau `freeze=0` |
| Loss tidak turun | Kurangi `lr0=0.0005` |
| Overfitting | Tambah `weight_decay=0.001` |
| OOM | Kurangi `batch=8`, `imgsz=416` |
| Dataset kecil (1734) | Naikkan augmentasi: `mosaic=1.0, copy_paste=0.2` |